# Fine-tune Qwen2.5-3B-Instruct on Mental Health Conversations with QLoRA + Unsloth

This notebook fine-tunes Qwen2.5-3B-Instruct on the [ShivomH/Mental-Health-Conversations](https://huggingface.co/datasets/ShivomH/Mental-Health-Conversations) dataset using QLoRA via Unsloth.

**Output**: 3 versions pushed to `nomeda-lab/nomeda-therapist`:
1. `bf16` – merged full-precision model
2. `8bit` – 8-bit quantized
3. `4bit` – 4-bit quantized

## 1. Install Dependencies

In [4]:
%%capture
import torch
major, minor = torch.__version__.split('.')[:2]
if int(major) == 2 and int(minor) == 5:
    %uv pip install --no-deps unsloth
else:
    %uv pip install unsloth

%uv pip install --no-deps xformers trl peft accelerate bitsandbytes

In [5]:
%uv pip install  --no-deps xformers trl peft accelerate bitsandbytes unsloth

Using Python 3.12.6 environment at: /usr/local
Resolved 6 packages in 9ms
Installed 1 package in 85ms
 + unsloth==2026.5.2
Note: you may need to restart the kernel to use updated packages.


## 2. Login to Hugging Face

In [24]:
from huggingface_hub import notebook_login
notebook_login()

In [8]:
%uv pip install unsloth_zoo

Using Python 3.12.6 environment at: /usr/local
Resolved 78 packages in 74ms
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠹ Preparing packages... (0/1)
⠹ Preparing packages... (0/1)
⠹ Preparing packages... (0/1)
⠸ Preparing packages... (0/1)
⠸ Preparing packages... (0/1)
⠸ Preparing packages... (0/1)
⠸ Preparing packages... (0/1)
⠼ Preparing packages... (0/1)
⠼ Preparing packages... (0/1)
⠼ Preparing packages... 

## 3. Load Model & Tokenizer

In [13]:
import torch
from unsloth import FastLanguageModel
from transformers import TextStreamer

model_name = "Qwen/Qwen3-1.7B"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=True,
)

==((====))==  Unsloth 2026.5.2: Fast Qwen3 patching. Transformers: 4.56.0.
   \\   /|    NVIDIA A100 80GB PCIe. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu129. CUDA: 8.0. CUDA Toolkit: 12.9. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
unsloth/qwen3-1.7b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


## 4. Configure LoRA

In [14]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

## 5. Load & Format Dataset

The dataset has `messages` in OpenAI chat format. We convert each conversation into a single text string using the model's chat template.

In [11]:
from datasets import load_dataset
import re

# ── 1. Load ───────────────────────────────────────────────────────────────────
dataset = load_dataset("ShenLab/MentalChat16K", split="train")
print(f"Raw size: {len(dataset)}")
print(f"Columns: {dataset.column_names}")
print(f"Sample:\n{dataset[0]}\n")

# ── 2. Clean ──────────────────────────────────────────────────────────────────
def clean_text(text):
    if not isinstance(text, str):
        return None
    text = text.strip()
    # Remove excessive whitespace / newlines
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r" {2,}", " ", text)
    # Remove null-like strings
    if text.lower() in {"none", "null", "nan", "", "n/a"}:
        return None
    return text

MIN_INPUT_LEN  = 10   # user message must be at least 10 chars
MIN_OUTPUT_LEN = 20   # therapist response must be at least 20 chars
MAX_INPUT_LEN  = 2048
MAX_OUTPUT_LEN = 2048

def process(example):
    instruction = clean_text(example.get("instruction"))
    inp         = clean_text(example.get("input"))
    out         = clean_text(example.get("output"))

    # Validity flags
    valid = (
        instruction is not None and
        inp  is not None and len(inp)  >= MIN_INPUT_LEN  and len(inp)  <= MAX_INPUT_LEN  and
        out  is not None and len(out)  >= MIN_OUTPUT_LEN and len(out)  <= MAX_OUTPUT_LEN
    )

    return {
        "instruction": instruction or "",
        "input":       inp         or "",
        "output":      out         or "",
        "valid":       valid,
    }

dataset = dataset.map(process)
before  = len(dataset)
dataset = dataset.filter(lambda x: x["valid"])
dataset = dataset.remove_columns(["valid"])
after   = len(dataset)
print(f"Dropped {before - after} bad rows → {after} clean examples remaining")

# ── 3. Dedup on (input, output) ───────────────────────────────────────────────
seen   = set()
def is_unique(example):
    key = (example["input"][:200], example["output"][:200])
    if key in seen:
        return False
    seen.add(key)
    return True

dataset = dataset.filter(is_unique)
print(f"After dedup: {len(dataset)} examples")

# ── 4. Split ──────────────────────────────────────────────────────────────────
dataset       = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset["train"]
eval_dataset  = dataset["test"]
print(f"Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")

# ── 5. Apply chat template ────────────────────────────────────────────────────
def format_chat_template(example):
    messages = [
        {"role": "system",    "content": example["instruction"].strip()},
        {"role": "user",      "content": example["input"].strip()},
        {"role": "assistant", "content": example["output"].strip()},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

train_dataset = train_dataset.map(format_chat_template, remove_columns=["instruction", "input", "output"])
eval_dataset  = eval_dataset.map(format_chat_template,  remove_columns=["instruction", "input", "output"])

# ── 6. Sanity check ───────────────────────────────────────────────────────────
print("\n── Sample formatted example ──")
print(train_dataset[0]["text"][:600])
print("\n── Length stats ──")
lengths = [len(x["text"]) for x in train_dataset]
print(f"Min: {min(lengths)} | Max: {max(lengths)} | Avg: {int(sum(lengths)/len(lengths))}")

README.md: 0.00B [00:00, ?B/s]

Interview_Data_6K.csv:   0%|          | 0.00/13.6M [00:00<?, ?B/s]

Synthetic_Data_10K.csv:   0%|          | 0.00/32.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16084 [00:00<?, ? examples/s]

Raw size: 16084
Columns: ['instruction', 'input', 'output']
Sample:
{'instruction': "You are a helpful mental health counselling assistant, please answer the mental health questions based on the patient's description. \nThe assistant gives helpful, comprehensive, and appropriate answers to the user's questions. ", 'input': "I've been struggling with my mental health for a while now, and I can't seem to find a way to cope with it. I've tried visualization, positive thinking, and even medication, but nothing seems to work. I've been feeling lost and helpless, and I don't know what to do next. My mind is a whirlwind of thoughts and emotions, and I can't seem to make sense of it all. I feel like I'm drowning in a sea of confusion, and I can't seem to find my way out.", 'output': "I understand that you've been dealing with a sense of confusion and chaos in your thoughts and emotions for some time now. It's been a challenging journey, and it's commendable that you've tried various approaches

...

Filter:   0%|          | 0/6815 [00:00<?, ? examples/s]

After dedup: 6683 examples
Train: 6014 | Eval: 669


Map:   0%|          | 0/6014 [00:00<?, ? examples/s]

Map:   0%|          | 0/669 [00:00<?, ? examples/s]


── Sample formatted example ──
<|im_start|>system
You are a helpful mental health counselling assistant, please answer the mental health questions based on the patient's description. 
The assistant gives helpful, comprehensive, and appropriate answers to the user's questions.<|im_end|>
<|im_start|>user
I've been struggling to find the motivation to complete my income tax return. I've always put it off until the last minute, but this year, I've been finding it particularly difficult. I've even considered asking for an extension, but I don't think I need one. I'm just not sure what to do.<|im_end|>
<|im_start|>assistant
<thin

── Length stats ──
Min: 800 | Max: 4394 | Avg: 2084


## 7. Train

In [15]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

args = TrainingArguments(
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumultion_steps=4,
    warmup_steps=20,
    num_train_epochs=2,
    learning_rate=2e-4,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=500,
    output_dir="outputs",
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    report_to="none",
    # ── Fix for 'int has no attribute mean' bug ──
    average_tokens_across_devices=False,  # ← add this
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=4096,
    dataset_num_proc=2,
    packing=False,
    args=args,
)

trainer_stats = trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=47):   0%|          | 0/6014 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=47):   0%|          | 0/669 [00:00<?, ? examples/s]

[accelerate.utils.other|WARNING]Detected kernel version 4.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 6,014 | Num Epochs = 2 | Total steps = 752
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 17,432,576 of 1,738,007,552 (1.00% trained)


Step,Training Loss,Validation Loss
200,0.959700,0.963458
400,0.858600,0.919317
600,0.833500,0.901762


## 8. Save 3 Versions to Hugging Face Hub

All three are pushed under `nomeda-lab/nomeda-therapist` with revision tags.

In [18]:
HF_USER = "nomeda-lab"
REPO_NAME = "nomeda-therapist-2B"
REPO_ID = f"{HF_USER}/{REPO_NAME}"

In [21]:
from huggingface_hub import HfApi
api = HfApi()
api.create_repo(repo_id=REPO_ID, private=False, exist_ok=True)

RepoUrl('https://huggingface.co/nomeda-lab/nomeda-therapist-2B', endpoint='https://huggingface.co', repo_type='model', repo_id='nomeda-lab/nomeda-therapist-2B')

### 8a. BF16 (merged, full precision)

In [22]:
model.save_pretrained_merged(
    "model_bf16",
    tokenizer,
    save_method="merged_16bit",
)

model.push_to_hub_merged(
    repo_id=REPO_ID,
    tokenizer=tokenizer,
    save_method="merged_16bit",
    revision="bf16",
)

config.json:   0%|          | 0.00/752 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|                                         | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|█████████████████████████████████| 1/1 [00:12<00:00, 12.29s/it]


...

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  ...meda-therapist-2B/model.safetensors:   3%|2         | 95.1MB / 3.44GB            

Unsloth: Merging weights into 16bit: 100%|███████████████████████████████████████| 1/1 [00:26<00:00, 26.25s/it]


Unsloth: Merge process complete. Saved to `/root/nomeda-lab/nomeda-therapist-2B`


In [26]:
import torch
from huggingface_hub import HfApi
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer, BitsAndBytesConfig

api = HfApi()
HF_USER   = "nomeda-lab"
REPO_FP8  = f"{HF_USER}/nomeda-therapist-2B-fp8"
REPO_Q4   = f"{HF_USER}/nomeda-therapist-2B-Q4"

api.create_repo(repo_id=REPO_FP8, private=False, exist_ok=True)
api.create_repo(repo_id=REPO_Q4,  private=False, exist_ok=True)

# ── First: save the LoRA adapter locally ─────────────────────────────────────
model.save_pretrained("lora_adapter")
tokenizer.save_pretrained("lora_adapter")
print("✅ LoRA adapter saved")

# ── 8-bit merged model ────────────────────────────────────────────────────────
bnb_8bit = BitsAndBytesConfig(load_in_8bit=True)

model_8bit = AutoPeftModelForCausalLM.from_pretrained(
    "lora_adapter",
    quantization_config=bnb_8bit,
    device_map="auto",
    torch_dtype=torch.float16,
)
model_8bit = model_8bit.merge_and_unload()
model_8bit.save_pretrained("model_8bit")
tokenizer.save_pretrained("model_8bit")

api.upload_folder(folder_path="model_8bit", repo_id=REPO_FP8,
                  commit_message="Add merged 8bit model")
print(f"✅ FP8 → https://huggingface.co/{REPO_FP8}")

# ── Clean up 8bit from memory before loading 4bit ────────────────────────────
del model_8bit
torch.cuda.empty_cache()

# ── 4-bit merged model ────────────────────────────────────────────────────────
bnb_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model_4bit = AutoPeftModelForCausalLM.from_pretrained(
    "lora_adapter",
    quantization_config=bnb_4bit,
    device_map="auto",
    torch_dtype=torch.float16,
)
model_4bit = model_4bit.merge_and_unload()
model_4bit.save_pretrained("model_4bit")
tokenizer.save_pretrained("model_4bit")

api.upload_folder(folder_path="model_4bit", repo_id=REPO_Q4,
                  commit_message="Add merged 4bit model")
print(f"✅ Q4  → https://huggingface.co/{REPO_Q4}")

`torch_dtype` is deprecated! Use `dtype` instead!


✅ LoRA adapter saved


/usr/local/lib/python3.12/site-packages/peft/tuners/lora/bnb.py:373: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

...

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  /root/model_4bit/tokenizer.json       : 100%|##########| 11.4MB / 11.4MB            

  /root/model_4bit/model.safetensors    :   7%|6         | 92.2MB / 1.41GB            

✅ Q4  → https://huggingface.co/nomeda-lab/nomeda-therapist-2B-Q4


## 9. Inference Test

Quick sanity check with a sample query.

In [27]:
messages = [
    {"role": "system", "content": "You are Nomeda the compassionate and caring mental health AI. Communicate with warmth, empathy, and professionalism."},
    {"role": "user", "content": "I've been feeling really anxious about my upcoming exams. Any advice?"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=256,
    temperature=0.7,
    top_p=0.9,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=False))

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


<|im_start|>system
You are Nomeda the compassionate and caring mental health AI. Communicate with warmth, empathy, and professionalism.<|im_end|>
<|im_start|>user
I've been feeling really anxious about my upcoming exams. Any advice?<|im_end|>
<|im_start|>assistant
<think>

</think>

I can understand how anxiety can be overwhelming during exam periods. It's natural to feel nervous and worried about the outcome. Here are some suggestions that may help alleviate some of the stress:

1. Create a study schedule: Break down your study material into manageable chunks and assign specific times each day for different subjects or topics. This will give you a clear plan and reduce the feeling of being overwhelmed.

2. Practice relaxation techniques: Engage in activities that help you relax and reduce stress, such as deep breathing exercises, meditation, or listening to calming music. Taking short breaks throughout your study sessions can also help refresh your mind.

3. Get enough sleep: Prioriti

In [2]:
import torch
import time
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

HF_USER   = "nomeda-lab"
REPO_BF16 = f"{HF_USER}/nomeda-therapist-2B"
REPO_FP8  = f"{HF_USER}/nomeda-therapist-2B-fp8"
REPO_Q4   = f"{HF_USER}/nomeda-therapist-2B-Q4"

# ── Kill the current Unsloth model from memory first ─────────────────────────
torch.cuda.empty_cache()
print("✅ Cleared Unsloth model from memory")

TEST_MESSAGES = [
    {"role": "system", "content": "You are Nomeda, a compassionate mental health AI. Give SHORT, warm responses — 2-3 sentences max."},
    {"role": "user",   "content": "I've been feeling really anxious about my upcoming exams. Any advice?"},
]

def load_and_test(repo_id, label, bnb_config=None):
    print(f"\n{'='*55}")
    print(f"  {label}")
    print(f"{'='*55}")

    tok = AutoTokenizer.from_pretrained(
        repo_id, trust_remote_code=True
    )

    kwargs = dict(
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.float16,
    )
    if bnb_config:
        kwargs["quantization_config"] = bnb_config

    # ── Pure HuggingFace load — NO Unsloth ───────────────────────────────────
    mdl = AutoModelForCausalLM.from_pretrained(repo_id, **kwargs)
    mdl.eval()

    # VRAM
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        print(f"  GPU {i}: {(total-free)/1e9:.2f}GB used / {total/1e9:.2f}GB total")

    # Tokenize
    inputs = tok.apply_chat_template(
        TEST_MESSAGES,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    t0 = time.time()
    with torch.no_grad():
        out = mdl.generate(
            input_ids=inputs,
            max_new_tokens=80,
            min_new_tokens=20,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
            pad_token_id=tok.eos_token_id,
        )
    elapsed = time.time() - t0
    n_tokens = out.shape[1] - inputs.shape[1]

    response = tok.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f"  ⏱  {elapsed:.2f}s | ⚡ {n_tokens/elapsed:.1f} tok/s | 🔢 {n_tokens} tokens")
    print(f"\n  💬 {response.strip()}\n")

    del mdl
    torch.cuda.empty_cache()

    return {"label": label, "time": elapsed,
            "tokens": n_tokens, "speed": n_tokens/elapsed,
            "response": response.strip()}

# ── Run all 3 ─────────────────────────────────────────────────────────────────
results = []

results.append(load_and_test(REPO_BF16, "BF16 — Full Precision"))

results.append(load_and_test(
    REPO_FP8, "FP8 — 8-bit",
    bnb_config=BitsAndBytesConfig(load_in_8bit=True)
))

results.append(load_and_test(
    REPO_Q4, "Q4 — 4-bit",
    bnb_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
))

# ── Summary ───────────────────────────────────────────────────────────────────
print("\n" + "="*55)
print("  COMPARISON SUMMARY")
print("="*55)
print(f"  {'Model':<22} {'Time':>7} {'Speed':>10} {'Tokens':>8}")
print(f"  {'-'*22} {'-'*7} {'-'*10} {'-'*8}")
for r in results:
    print(f"  {r['label']:<22} {r['time']:>6.2f}s {r['speed']:>9.1f}t/s {r['tokens']:>8}")
print("="*55)
print("\n  RESPONSES:")
for r in results:
    print(f"\n  [{r['label']}]")
    print(f"  {r['response']}")

✅ Cleared Unsloth model from memory

  BF16 — Full Precision


`torch_dtype` is deprecated! Use `dtype` instead!
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.8.0+cu129).
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


  GPU 0: 3.99GB used / 85.09GB total
  ⏱  2.66s | ⚡ 30.1 tok/s | 🔢 80 tokens

  💬 <think>

</think>

It's completely normal to feel nervous about exams! Remember that you're not alone in this experience. Anxiety can often be overwhelming, but there are strategies that may help alleviate some of the pressure.

Firstly, try breaking down your study sessions into smaller tasks. This way, it will be easier for you to focus and manage the workload. Additionally, taking regular breaks throughout your study


  FP8 — 8-bit


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.41G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

  GPU 0: 4.16GB used / 85.09GB total
  ⏱  3.95s | ⚡ 20.2 tok/s | 🔢 80 tokens

  💬 <think>
Okay, the user is anxious about exams and needs advice. First, I should acknowledge their feelings to show empathy. Maybe start with something like "It's completely normal to feel anxious about exams." Then, offer practical tips. Break down the steps into manageable tasks, like creating a study schedule. Also, remind them to take care of self-care, like exercise or sleep. End with encouragement


  COMPARISON SUMMARY
  Model                     Time      Speed   Tokens
  ---------------------- ------- ---------- --------
  BF16 — Full Precision    2.66s      30.1t/s       80
  FP8 — 8-bit              4.04s      19.8t/s       80
  Q4 — 4-bit               3.95s      20.2t/s       80

  RESPONSES:

  [BF16 — Full Precision]
  <think>

</think>

It's completely normal to feel nervous about exams! Remember that you're not alone in this experience. Anxiety can often be overwhelming, but there are stra

---
**Done!** The model `nomeda-lab/nomeda-therapist` now has 3 revisions: `bf16`, `8bit`, and `4bit`.